# Ejercicio 3 - Validacion y calidad de datos 

En este notebook se ejecutaran las funciones pedidas en el ejercicio 3 para verificar su correcto funcionamiento
Se utilizara como ejemplo el dataset iNaturalist

In [1]:
from pathlib import Path
import sys

DIC_BASE = Path.cwd().resolve().parent
sys.path.insert(0,str(DIC_BASE))


from datetime import datetime
import csv
import pycountry
from src.lectura import count_records

file_route = DIC_BASE / 'raw_datasets' / 'inaturalist-filtered' / 'observations.csv'

### Ejercicio 3.A

Los campos decimalLatitude|latitudeDecimal,decimalLongitude|longitudeDecimal deben representar coordenadas geográficas válidas.

Escribir una función que detecte registros donde:
 - decimalLatitude no esté en el rango [-90, 90]
 - decimalLongitude no esté en el rango [-180, 180]
 - alguno de los valores no pueda convertirse a número

La función debe retornar:
 - cantidad de registros inválidos​
 - listado de registros incorrectos.

In [2]:
from src.validaciones import validar_coordenadas

resultado = validar_coordenadas(path = file_route, dataset = 'inaturalist')
print(f"La cantidad de registro invalidos son: {resultado['cantidad_invalidos']}")
for registro in resultado['lista_invalidos']: print(f"ID del registro: {registro}")

La cantidad de registro invalidos son: 2
ID del registro: 999990001
ID del registro: 999990002


### Ejercicio 3.B

Un registro geográfico debería tener ambas coordenadas. Escribir una función que detecte
registros donde:
- exista decimalLatitude pero no decimalLongitude
- exista decimalLongitude pero no decimalLatitude.

In [3]:
from src.validaciones import constatar_coordenadas

resultado = constatar_coordenadas(path = file_route, dataset = 'inaturalist')
print(f"La cantidad de registro con inconsistencias son: {resultado['cantidad_inconsistencias']}")
for registro in resultado['lista_ids']: print(f"ID del registro:{registro}")

Evaluando inconsistencias en las coordenadas...
La cantidad de registro con inconsistencias son: 0


### Ejercicio 3.C

Escribir una función que detecte:
- fechas con formato inválido​
- fechas que no puedan interpretarse como fecha​
- fechas posteriores al año actual

In [4]:
from src.validaciones import validar_fechas

resultado = validar_fechas(path = file_route, dataset = 'inaturalist')
print(f"La cantidad de fechas posteriores a 2026 son: {resultado['anios_posteriores']}")
print(f"La cantidad de fechas invalidas son: {resultado['fechas_invalidas']}")

Evaluando fechas del dataset...
La cantidad de fechas posteriores a 2026 son: 0
La cantidad de fechas invalidas son: 0


### Ejercicio 3.D

Utilizando los “ID” escribir una función que detecte posibles registros duplicados.
La función deberá indicar:
- cuántos registros duplicados existen​
- qué identificador se repite.​

In [5]:
from src.validaciones import verificar_duplicados

resultado = verificar_duplicados(path = file_route, dataset = 'inaturalist')
print(f"La cantidad de registros duplicados son: {resultado['cantidad_duplicados']}")
for registro in resultado['id_duplicados']: print(f"ID del registro: {registro}")

Evaluando registros repetidos del dataset...
La cantidad de registros duplicados son: 1
ID del registro: 42369866


### Ejercicio 3.E

El campo countryCode puede tomar solo un conjunto de valores específicos, investigar
cuáles son y desarrollar una función que detecte valores no permitidos.

In [6]:
from src.validaciones import verificar_countryCode

resultado = verificar_countryCode(path = file_route, dataset = 'inaturalist')
for registro in resultado['lista_ids']: print(f"ID del registro: {registro}")

Evaluando errores en el campo 'countryCode' del dataset...
El codigo ZZ no es valido
ID del registro: 999990003


### Ejercicio 3.F

El campo coordinateUncertaintyInMeters representa la incertidumbre en la definición de la
coordenada. 

Escribir una función que detecte registros donde:
- el valor no sea numérico
- el valor sea negativo
- el valor sea excesivamente grande (por ejemplo > 100).

In [7]:
from src.validaciones import verificar_incertidumbre

resultado = verificar_incertidumbre(path = file_route, dataset = 'inaturalist')
print(f"La cantidad de datos no validos son: {resultado['dato_invalido']}")
print(f"La cantidad de datos no numeros son: {resultado['no_dato']}")
print(f"Primeras IDS: {resultado['lista_ids'][:10]}")

Evaluando errores en el campo 'coordinateUncertainyInMeters' del dataset...
La cantidad de datos no validos son: 581648
La cantidad de datos no numeros son: 227702
Primeras IDS: ['254213783', '254213786', '254213816', '254213853', '254213899', '127106901', '254214031', '254214074', '254214110', '254214178']


### Ejercicio 3.G

Escribir una función que genere un resumen de calidad del dataset, incluyendo:
- cantidad total de registros
- registros con coordenadas inválidas
- registros con fechas inválidas
- registros duplicados
- registros con información taxonómica incompleta
  
El resultado puede ser un diccionario y un pequeño reporte en texto.

In [8]:
from src.validaciones import resumen_calidad

dicc = resumen_calidad(path = file_route, dataset = 'inaturalist')
print("")
print(f"Diccionario devuelto: {dicc}")

Evaluando fechas del dataset...
Evaluando registros repetidos del dataset...
Buscando errores en los datos taxonomicos...

**************************************************
RESUMEN DE CALIDAD DEL DATASET inaturalist
**************************************************
Cantidad total de registros analizados: 1229610
--------------------------------------------------
Cantidad de errores en las coordenadas 'latitud' y 'longitud': 2
Cantidad de errores en las fechas: 0
Cantidad de datos duplicados: 1
Cantidad de registros con informacion taxonomica incompleta: 759

Diccionario devuelto: {'registro': 1229610, 'error_coordenadas': 2, 'error_fechas': 0, 'duplicados': 1, 'error_taxonomico': 759}


### Ejercicio 3.H

Establezca cotas de coordenadas para el américa del sur, es decir latitudes y longitudes
máximas y mínimas que delimitan ocurrencias dentro del territorio (aproximado). A partir de
estas cotas seteadas como constantes escriba una función que valide los campos de latitud
y longitud de cada dataset.

In [9]:
from src.validaciones import evaluar_cotas_america

resultado = evaluar_cotas_america(path = file_route, dataset = 'inaturalist')
print(f"Cantidad de datos invalidos para la latitud: {resultado['latitudes_invalidas']}")
print(f"Cantidad de datos invalidos para la longitud: {resultado['longitudes_invalidas']}")
for registro in resultado['lista_ids']: print(f"ID del registro: {registro}")

Evaluando cotas de coordenadas (America del sur) del dataset...
Cantidad de datos invalidos para la latitud: 1
Cantidad de datos invalidos para la longitud: 1
ID del registro: 999990001
ID del registro: 999990002


### Ejercicio 3.I

Escriba una función que agrupe y ejecute todas las validaciones para el campo latitud, y otra
independiente para el campo longitud.

In [10]:
from src.validaciones import validar_longitud,validar_latitud

resultado_lat = validar_latitud(path = file_route, dataset = 'inaturalist')
resultado_lon = validar_longitud(path = file_route, dataset = 'inaturalist')

print(f"La cantidad de latitudes invalidas son : {resultado_lat['resultado_cotas']['latitudes_invalidas']}")
print(f"La cantidad de coordenadas invalidas son : {resultado_lat['resultado_coordenadas']['cantidad_invalidos']}")
print("")
print(f"La cantidad de longitudes invalidas son : {resultado_lon['resultado_cotas']['longitudes_invalidas']}")
print(f"La cantidad de coordenadas invalidas son : {resultado_lon['resultado_coordenadas']['cantidad_invalidos']}")

Evaluando cotas de coordenadas (America del sur) del dataset...
Evaluando cotas de coordenadas (America del sur) del dataset...
La cantidad de latitudes invalidas son : 1
La cantidad de coordenadas invalidas son : 1

La cantidad de longitudes invalidas son : 1
La cantidad de coordenadas invalidas son : 1
